# MOSS-TTS-Realtime Quick Start

This notebook demonstrates the basic usage of [MOSS-TTS-Realtime](https://github.com/OpenMOSS/MOSS-TTS), a context-aware, multi-turn streaming TTS foundation model designed for real-time voice agents.

**Contents:**
1. Installation
2. Option A — Gradio Demo (interactive web UI)
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Basic Usage (Non streaming)


## 1. Installation

Clone the repository and install dependencies.

In [ ]:
!git clone https://github.com/leewinn1/MOSS-TTS.git
%cd MOSS-TTS
!pip install --extra-index-url https://download.pytorch.org/whl/cu128 -e .
%cd moss_tts_realtime

## 2. Option A — Gradio Demo

Launch an interactive web UI with a public Gradio link. The `--share` flag creates a temporary public URL so you can access the demo from any browser.

> **If you prefer to use the Python API directly, skip to Option B below.**

In [ ]:
!python3 app.py --share

## 3. Option B — Python API

### 3.1 Load Model

In [ ]:
import importlib.util
import torch
import torchaudio
from transformers import AutoTokenizer, AutoModel
from mossttsrealtime.modeling_mossttsrealtime import MossTTSRealtime
from inferencer import MossTTSRealtimeInference

CODEC_SAMPLE_RATE = 24000
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MossTTSRealtime.from_pretrained("OpenMOSS-Team/MOSS-TTS-Realtime", attn_implementation="sdpa", torch_dtype=torch.bfloat16).to(device)
tokenizer = AutoTokenizer.from_pretrained("OpenMOSS-Team/MOSS-TTS-Realtime")
codec = AutoModel.from_pretrained("OpenMOSS-Team/MOSS-Audio-Tokenizer", trust_remote_code=True).eval().to(device)

inferencer = MossTTSRealtimeInference(model, tokenizer, max_length=5000, codec=codec, codec_sample_rate=CODEC_SAMPLE_RATE, codec_encode_kwargs={"chunk_duration": 8})


### 3.2 Basic Usage

Provide text and reference audio paths. We can use the default prompt audio provided in the repo.

In [ ]:
text = ["Welcome to the world of MOSS TTS Realtime. Experience how text transforms into smooth, human-like speech in real time."]
reference_audio_path = ["./audio/prompt_audio1.mp3"]


In [ ]:
from IPython.display import Audio, display

result = inferencer.generate(
    text=text,
    reference_audio_path=reference_audio_path,
    temperature=0.8,
    top_p=0.6,
    top_k=30,
    repetition_penalty=1.1,
    repetition_window=50,
    device=device,
)

for i, generated_tokens in enumerate(result):
    output = torch.tensor(generated_tokens).to(device)
    decode_result = codec.decode(output.permute(1, 0), chunk_duration=8)
    wav = decode_result["audio"][0].cpu().detach()
    display(Audio(wav.numpy(), rate=CODEC_SAMPLE_RATE))
